# Performing Domain Analysis with AI  
## Leveraga SA Model, Write Up, and Results on Strategy to develop a first cut Architecture

First we load up the PEAK Model

### Model-Specific Code (Do Not Modify)

This section contains code that is specific to the system model. It is updated only when the model is changed and should not require user modifications under normal circumstances.

If a new model is introduced, ensure this section is reviewed and updated as needed.


In [1]:
#!pip install --upgrade git+https://github.com/tkSDISW/Capella_Tools 
import capellambse.decl

from capella_tools import capellambse_helper

from IPython import display as diag_display
resources = {
    "PEAK": "PEAK/PEAK",
}
path_to_model = "../PEAK.aird"
model = capellambse.MelodyModel(path_to_model, resources=resources)
from capella_tools import Pub4C
# Instantiate the class with the traceability file
traceability_store = Pub4C.Traceability_Store("../PEAK/traceability")


## 🔄 Embedding Generation Process

Next we load evaluate whether we ne to update the embedings. If there older than model we will update thems. 🚀


In [2]:

from capella_tools  import capella_embeddings_manager

# Generate embeddings for all objects
model_embedding_manager = capella_embeddings_manager.EmbeddingManager()

embedding_file = "embeddings.json" 
model_embedding_manager.set_files( path_to_model , embedding_file)

model_embedding_manager.create_model_embeddings(model)

OpenAI API Key retrieved successfully.
Loading embeddings
embeddings loaded.


## 🎯 Define the model element or diagram for analysis.

We use the embedding to locate a diagram that will be used for basis of analysis. It a OCB diagrams with relations to all the functional chains. 

> 💡 **Tip:** If you're unsure about the model structure, review the documentation or refer to the model's diagrams for additional guidance.


In [3]:

selected_objects = model_embedding_manager.query_and_select_top_objects("[LAB] Water Generation QAS and Params", top_n=1)



This is a list of ranked Objects Based on Query:
Index: 0, Name: [LAB] Water Generation QAS and Params, Similarity: 0.84, Type: Diagram, Phase: Logical Architecture LA, Source: , Target: 


## 📝 Generate Structured Input File 

We then generate the structured input file for all the matching objects and there related objects.
A file "capella_model.yaml" is written it you want to look at it. 

In [4]:
#Workflow
from capella_tools import capellambse_yaml_manager
#import capellambse_yaml_manager
yaml_handler = capellambse_yaml_manager.CapellaYAMLHandler()
   
#Generate YAML for the logical component and append to the file
for object in  selected_objects :  
    yaml_handler.generate_yaml(model.by_uuid(object["uuid"]))  

yaml_handler.generate_traceability_related_objects(model,traceability_store)

#yaml_handler.display()
yaml_handler.generate_yaml_referenced_objects()
#yaml_handler.display()

yaml_handler.write_output_file()


## ⚙️ Execute  Prompt


Execute the prompt that will use the model and system document. 

In [5]:
import os
from openai import OpenAI
from IPython.core.display import HTML
from IPython.display import display, clear_output, Markdown, IFrame
from capella_tools  import Open_AI_RAG_manager


#print(object)
# Step 1: Get YAML content
yaml_content = yaml_handler.get_yaml_content()

# Step 2: Invoke ChatGPT for analysis
analyzer = Open_AI_RAG_manager.ChatGPTAnalyzer(yaml_content)#analyzer.add_text_file_to_messages("PEAK System.docx")
prompt = """
Write image description for the visually impaired based on the content of this diagram. Specifically call attention to how the PEAK water subsystem  has been decomposed.
"""
analyzer.follow_up_prompt(prompt)
chatgpt_response = analyzer.get_response()

OpenAI API Key retrieved successfully.


**Your prompt:** 
Write image description for the visually impaired based on the content of this diagram. Specifically call attention to how the PEAK water subsystem  has been decomposed.


**ChatGPT Response:**

I apologize for any confusion, but I can't see or interpret diagrams or images directly. However, based on the information provided in the YAML file, I can describe the structure and context of the content related to the PEAK water subsystem. 

The YAML file details a system model focused on the relationship between various components, primarily including an object named "[LAB] Water Generation QAS and Params", which is of the type DRepresentationDescriptor. Although the YAML content doesn't explicitly mention a PEAK water subsystem or its decomposition, one could assume from the context that the PEAK water subsystem might be part of the water generation and parameter settings as described by the object.

In a typical system diagram, a subsystem like the PEAK water subsystem would be represented as a section or module within a larger system model. This kind of decomposition could show various components or processes involved in generating or managing water. Each component might be visually represented using different shapes or symbols, such as rectangles for processes, circles for events, and arrows to illustrate relationships or data flow between components.

For someone visually impaired, if this diagram relates to a PEAK water subsystem, it'd be important to highlight the hierarchical structure. A main system block labeled as "PEAK Water Subsystem" might be broken down into smaller, interconnected parts. These could include components for water sourcing, treatment, quality assurance, and distribution, each with its flow or dependency connections, presumably influenced by unique identifiers like "primary_uuid" and "ref_uuid" that govern their interaction and reference within the system.

Unfortunately, without direct access to the visual content, this interpretation remains speculative and serves to provide a general guide on what the depiction might entail, based on typical practices in system modeling and the given YAML description.

**Token Usage Info:**

Tokens used: prompt=150, completion=347, total=497

## 💬 Launch Interactive Chat on Structured Input




In [6]:
print("Done")

Done


# 